## Task 1: Data Ingestion & Exploration
### Objective
#### Load the dataset into PySpark and perform exploratory analysis.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, count, col, round, avg, sum

In [0]:
#load dataset
df = spark.read.csv("/Volumes/workspace/default/new_volume/BNPParibas_Data.csv",
                    header=True,
                    inferSchema=True
                    )

In [0]:
#Schema
df.printSchema()

In [0]:
#  Record Count
df.count()

In [0]:
# data Types
df.dtypes


In [0]:
#null values
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

In [0]:
#Duplicate Count
total_rows = df.count()
unique_rows = df.distinct().count()
Duplicate_Count = total_rows - unique_rows
Duplicate_Count


In [0]:
#summary statistics
df.describe().display()

## Task 2: ETL Pipeline Development

##### Extract

In [0]:
df = spark.read.csv("/Volumes/workspace/default/new_volume/BNPParibas_Data.csv",
                    header=True,
                    inferSchema=True
                    )

##### Transformation logic

###### Missing Value Treatment


In [0]:
df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in df.columns
]).display()

####### so, here there is no null value present.


##### Duplicate Removal


In [0]:
df = df.dropDuplicates()
display(df)

##### Data Type Conversion

In [0]:
df.printSchema()

###### Everything already appears to be in the correct data type.

###### Feature Engineering


In [0]:
df = df.withColumn(
    "avg_monthly_spend",
    when(col("tenure_months") > 0,
         round(col("total_charges") / col("tenure_months"),2)).otherwise(0)
)
display(df)

##### Aggregation

##### Average Charges by Contract Type

In [0]:
agg_df = df.groupBy("contract_type").agg(
    avg("monthly_charges").alias("avg_mothly_charges"),
    avg("total_charges").alias("avg_total_charges")
)
display(agg_df)


##### Customer Count by Internet Service

In [0]:
agg_df = df.groupBy("internet_service").agg(
    count("*").alias("customer_count")
)
display(agg_df)

##### Load Phase

In [0]:
df.write.mode("overwrite")\
    .parquet("/Volumes/workspace/default/new_volume/silver_layer")

In [0]:
silver_df = spark.read.parquet("/Volumes/workspace/default/new_volume/silver_layer",
                               header=True,
                               inferSchema=True)


##### Task 3: ELT Pipeline & Medallion Architecture

###### Objective
###### Implement Bronze-Silver-Gold architecture.



In [0]:
#### Bronze Layer
df.write.mode("overwrite")\
    .parquet("/Volumes/workspace/default/new_volume/bronze_layer")

##### Silver Layer

In [0]:
#missing value treatment
df.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in df.columns
]).display()

In [0]:
# Duplicate removal
df.dropDuplicates().display()

In [0]:
# Data type coversion
df.printSchema()

###### Everything already appears to be in the correct data type.

In [0]:
#feature engineering
df = df.withColumn(
    "avg_monthly_spend",
    when(col("tenure_months") > 0,
         round(col("total_charges") / col("tenure_months"),2)).otherwise(0)
)
display(df)

In [0]:
df.write.mode("overwrite")\
    .parquet("/Volumes/workspace/default/new_volume/silver_layer")

##### Gold Layer

In [0]:
# kpi 1: Churn Analysis by Contract Type
contract_kpi = (
    silver_df.groupBy("contract_type")
    .agg(avg("churn").alias("churn_rate"))
)
contract_kpi.display()

In [0]:
# Revenue Analysis by Internet Service
revenue_kpi = (
    silver_df.groupBy("internet_service")
    .agg(
        sum("total_charges").alias("total_revenue")
    )
)
revenue_kpi.display()

In [0]:
# Support Ticket Analysis
support_kpi = (
    silver_df.groupBy("contract_type")
    .agg(
        avg("support_tickets").alias("avg_support_tickets")
    )
)
support_kpi.display()

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/new_volume/gold/category_performance")

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/new_volume/gold/Regional_Analysis")

In [0]:
df.write.mode("overwrite").parquet("/Volumes/workspace/default/new_volume/gold/Customer_Analysis")

##### Task 4: PySpark + Pandas Integration
###### Objective
###### Demonstrate interoperability between Pandas and Spark.


In [0]:
pdf = silver_df.toPandas()
type(pdf)

###### Feature Engineering in Pandas


In [0]:
#Ratio Column
pdf["charge_ratio"] = (
    pdf["total_charges"]/pdf["monthly_charges"]
)
pdf["charge_ratio"]

In [0]:
#Percentage Column
pdf["charge_percentage"] = (
    pdf["total_charges"]/pdf["monthly_charges"]*100
)
pdf["charge_percentage"]


In [0]:
#Growth Metric
pdf["growth_matric"] = ((pdf["monthly_charges"]-pdf["monthly_charges"].mean())/pdf["monthly_charges"].mean())*100
pdf["growth_matric"]

In [0]:
spark_df=spark.createDataFrame(pdf)
spark_df

##### Task 5: Spark SQL
###### Objective
####### Generate analytical reports using Spark SQL.


In [0]:
silver_df.createOrReplaceTempView("customers")

In [0]:
# Query 1: Top Contract Types by Customer Count
spark.sql("""
SELECT
    contract_type,
    COUNT(*) AS customer_count
FROM customers
GROUP BY contract_type
ORDER BY customer_count DESC
""").show()

In [0]:
#Query 2: Highest Revenue Segment
spark.sql("""
SELECT
    internet_service,
    SUM(total_charges) AS total_revenue
FROM customers
GROUP BY internet_service
ORDER BY total_revenue DESC
""").show()

In [0]:
#Query 3: Average Metric by Group
spark.sql("""
SELECT
    contract_type,
    ROUND(AVG(monthly_charges),2) AS avg_monthly_charges
FROM customers
GROUP BY contract_type
""").show()


In [0]:
#Query 4: Churn Analysis by Contract Type
spark.sql("""
SELECT
    contract_type,
    ROUND(100*SUM(churn)/COUNT(*),2) AS churn_rate
FROM customers
GROUP BY contract_type
ORDER BY churn_rate DESC""").show()


In [0]:
#Query 5: Top 10 Customers by Revenue
spark.sql("""
SELECT
    customer_id,
    total_charges
FROM customers
ORDER BY total_charges DESC
LIMIT 10
""").show()


###### Task 6: Advanced Transformations


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead

In [0]:
window_spec = Window.partitionBy("contract_type") \
                    .orderBy(col("total_charges").desc())

In [0]:
#row_number
silver_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

In [0]:
#rank()
silver_df.withColumn("rank",rank().over(window_spec))

In [0]:
#dense_rank()
silver_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
)

In [0]:
#lag()
silver_df.withColumn(
    "prev_charges",lag("total_charges",1).over(window_spec)
)

In [0]:
#lead
silver_df.withColumn(
    "next_charges",lead("total_charges",1).over(window_spec)
)

In [0]:
lookup_table=[("Month-to-Month","Fiber"),("One Year","None"),("Two Year","DSL")]
lookup_table = spark.createDataFrame(lookup_table,["contract_service","internet_service"])


In [0]:
#Inner join
inner_df=silver_df.join(lookup_table,on="contract_type",how="inner")


In [0]:
#left join
left_join = silver_df.join(lookup_table,on="contract_type",how="left")

##### Task 8: Performance Optimization


In [0]:
#Cache
silver_df.cache()
silver_df.count()

In [0]:
#Persist
from pyspark import StorageLevel
silver_df.persist(StorageLevel.MEMEORY_AND_DISK)
silver_df.count()


In [0]:
silver_df = silver_df.partition(8)

In [0]:
#Broadcast Join
from pyspark.sql.functions import broadcast

df.join(
    broadcast(lookup_df),
    on="internet_service",
    how="inner"
)